In [1]:
from pathlib import Path
import os
import time
import pandas as pd
from tqdm.auto import tqdm

import torch
from datasets import load_from_disk
from transformers import pipeline

In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

PROJECT_ROOT = Path("/home/ridwan/projects/afrispeech-project")

CLINICAL_ARROW_PATH = PROJECT_ROOT / "data" / "processed" / "clinical_test_arrow"
GENERAL_ARROW_PATH = PROJECT_ROOT / "data" / "processed" / "general_test_arrow"

PREDICTIONS_DIR = PROJECT_ROOT / "results" / "predictions"
LOGS_DIR = PROJECT_ROOT / "logs"

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Clinical path exists:", CLINICAL_ARROW_PATH.exists())
print("General path exists:", GENERAL_ARROW_PATH.exists())
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Project root: /home/ridwan/projects/afrispeech-project
Clinical path exists: True
General path exists: True
CUDA available: True
GPU: NVIDIA GeForce RTX 4090


In [3]:
clinical_ds = load_from_disk(str(CLINICAL_ARROW_PATH))
general_ds = load_from_disk(str(GENERAL_ARROW_PATH))

print("Clinical dataset:", clinical_ds)
print("General dataset:", general_ds)

print("Clinical samples:", len(clinical_ds))
print("General samples:", len(general_ds))

print("Columns:", clinical_ds.column_names)

Clinical dataset: Dataset({
    features: ['speaker_id', 'path', 'audio_id', 'audio', 'transcript', 'age_group', 'gender', 'accent', 'domain', 'country', 'duration'],
    num_rows: 3606
})
General dataset: Dataset({
    features: ['speaker_id', 'path', 'audio_id', 'audio', 'transcript', 'age_group', 'gender', 'accent', 'domain', 'country', 'duration'],
    num_rows: 2712
})
Clinical samples: 3606
General samples: 2712
Columns: ['speaker_id', 'path', 'audio_id', 'audio', 'transcript', 'age_group', 'gender', 'accent', 'domain', 'country', 'duration']


In [9]:
# ── Pipeline setup ──────────────────────────────────────────────────────────
MODEL_NAME = "openai/whisper-small"
MODEL_SHORT_NAME = "whisper_small"

device = 0 if torch.cuda.is_available() else -1
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

asr_pipe = pipeline(
    task="automatic-speech-recognition",
    model=MODEL_NAME,
    torch_dtype=torch_dtype,
    device=device,
    chunk_length_s=30,        # handles clips near 30s boundary cleanly
    stride_length_s=5,        # 5s overlap between chunks for long clips
)

print("Loaded model:", MODEL_NAME)
print("Device:", "cuda" if device == 0 else "cpu")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Loaded model: openai/whisper-small
Device: cuda


In [5]:
sample = clinical_ds[1]

print("Audio ID:", sample["audio_id"])
print("Domain:", sample["domain"])
print("Accent:", sample["accent"])
print("Country:", sample["country"])
print("Duration:", sample["duration"])
print("Reference:", sample["transcript"])

audio_input = {
    "array": sample["audio"]["array"],
    "sampling_rate": sample["audio"]["sampling_rate"]
}

prediction = asr_pipe(
    audio_input,
    return_timestamps = True,
    generate_kwargs={
        "language": "english",
        "task": "transcribe"
    }
)

print("\nPrediction:")
print(prediction["text"])

A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


Audio ID: 80581f4f-9e72-48fb-b4c0-22e6972dbff3/95ddccb7b9160663e9c7d36d3b275120
Domain: clinical
Accent: setswana
Country: ZA
Duration: 4.526984214782715
Reference: Ask the patient about allergies totape and skin antiseptics.


Prediction:
 Ask the patients about allergies to tape and skin antiseptics.


In [10]:
# ── Batched inference function ───────────────────────────────────────────────
def run_whisper_inference(
    dataset,
    output_csv_path,
    model_name,
    model_short_name,
    domain_name,
    max_samples=None,
    batch_size=16,
    flush_every=200
):
    """
    Run batched Whisper inference on a HuggingFace Arrow dataset.
    Saves predictions incrementally to CSV.
    """
    output_csv_path = Path(output_csv_path)
    n_samples = len(dataset) if max_samples is None else min(max_samples, len(dataset))

    # Build audio input list and metadata list separately
    # Avoids loading all audio arrays into memory at once
    def audio_generator():
        for i in range(n_samples):
            sample = dataset[i]
            yield {
                "array": sample["audio"]["array"],
                "sampling_rate": sample["audio"]["sampling_rate"]
            }

    # Collect metadata (no audio arrays — fast)
    print("Collecting metadata...")
    meta_rows = []
    for i in range(n_samples):
        sample = dataset[i]
        meta_rows.append({
            "row_index": i,
            "audio_id": sample.get("audio_id", ""),
            "speaker_id": sample.get("speaker_id", ""),
            "domain": sample.get("domain", ""),
            "accent": sample.get("accent", ""),
            "country": sample.get("country", ""),
            "gender": sample.get("gender", ""),
            "age_group": sample.get("age_group", ""),
            "duration": sample.get("duration", ""),
            "model_name": model_name,
            "model_short_name": model_short_name,
            "reference": sample.get("transcript", ""),
        })

    # Run batched inference
    print(f"Running inference on {n_samples} samples (batch_size={batch_size})...")
    start_time = time.time()

    predictions = []
    errors = []

    results = asr_pipe(
        audio_generator(),
        batch_size=batch_size,
        generate_kwargs={
            "language": "english",
            "task": "transcribe"
        }
    )

    for i, result in enumerate(tqdm(results, total=n_samples,
                                     desc=f"{model_short_name} | {domain_name}")):
        try:
            predictions.append(result["text"])
            errors.append("")
        except Exception as e:
            predictions.append("")
            errors.append(str(e))

        # Flush partial results every flush_every samples
        if (i + 1) % flush_every == 0:
            partial_df = pd.DataFrame(meta_rows[:i+1])
            partial_df["prediction"] = predictions
            partial_df["error"] = errors
            partial_df.to_csv(output_csv_path, index=False)
            print(f"  Flushed {i+1} rows to {output_csv_path}")

    # Final save
    df = pd.DataFrame(meta_rows)
    df["prediction"] = predictions
    df["error"] = errors
    df.to_csv(output_csv_path, index=False)

    elapsed = round((time.time() - start_time) / 60, 2)
    print(f"\nDone. Saved {len(df)} rows to {output_csv_path}")
    print(f"Elapsed: {elapsed} minutes")
    print(f"Errors: {df['error'].ne('').sum()}")

    return df

In [11]:
# Quick sanity check — 5 samples before full run
test_output_path = PREDICTIONS_DIR / "test_whisper_small_clinical_5samples.csv"

test_df = run_whisper_inference(
    dataset=clinical_ds,
    output_csv_path=test_output_path,
    model_name=MODEL_NAME,
    model_short_name=MODEL_SHORT_NAME,
    domain_name="clinical",
    max_samples=5,
    batch_size=4,
    flush_every=5
)

display(test_df[["audio_id", "accent", "reference", "prediction", "error"]])

Running inference on 5 samples (batch_size=4)...


whisper_small | clinical:   0%|          | 0/5 [00:00<?, ?it/s]

  Flushed 5 rows to /home/ridwan/projects/afrispeech-project/results/predictions/test_whisper_small_clinical_5samples.csv

Done. Saved 5 rows to /home/ridwan/projects/afrispeech-project/results/predictions/test_whisper_small_clinical_5samples.csv
Elapsed: 0.01 minutes
Errors: 0


,audio_id,accent,reference,prediction,error
0,27a83595-3d3f-4a6b-b909-7b8f364d736b/7cb8dfb31...,setswana,Protection of the host immune mechanism mighti...,Protection of the host immune mechanism might...,
1,80581f4f-9e72-48fb-b4c0-22e6972dbff3/95ddccb7b...,setswana,Ask the patient about allergies totape and ski...,Ask the patients about allergies to tape and ...,
2,13f2d282-bda1-4bc3-83b8-7a0d06e5eda0/933137f33...,setswana,"The short, mucus-secreting cells between the c...",The short macro secreting cells between the c...,
3,99f6019b-4505-4c60-a462-79b0eb6e689a/2cb25e2dd...,setswana,Aspiration is terminated when aspirated materi...,6. Aspiration is terminated when aspirated ma...,
4,13600ba0-a4d5-46f7-9235-776cd60bfdee/1404f622c...,setswana,Identify the appropriate landmarks for the sit...,Identify the appropriate landmarks for the si...,


In [13]:
clinical_output_path = PREDICTIONS_DIR / "whisper_small_clinical_predictions.csv"

clinical_df = run_whisper_inference(
    dataset=clinical_ds,
    output_csv_path=clinical_output_path,
    model_name=MODEL_NAME,
    model_short_name=MODEL_SHORT_NAME,
    domain_name="clinical",
    max_samples=None,
    batch_size=32,
    flush_every=200
)

display(clinical_df[["audio_id", "accent", "reference", "prediction", "error"]].head(10))

Running inference on 3606 samples (batch_size=32)...


whisper_small | clinical:   0%|          | 0/3606 [00:00<?, ?it/s]

  Flushed 200 rows to /home/ridwan/projects/afrispeech-project/results/predictions/whisper_small_clinical_predictions.csv
  Flushed 400 rows to /home/ridwan/projects/afrispeech-project/results/predictions/whisper_small_clinical_predictions.csv
  Flushed 600 rows to /home/ridwan/projects/afrispeech-project/results/predictions/whisper_small_clinical_predictions.csv
  Flushed 800 rows to /home/ridwan/projects/afrispeech-project/results/predictions/whisper_small_clinical_predictions.csv
  Flushed 1000 rows to /home/ridwan/projects/afrispeech-project/results/predictions/whisper_small_clinical_predictions.csv
  Flushed 1200 rows to /home/ridwan/projects/afrispeech-project/results/predictions/whisper_small_clinical_predictions.csv
  Flushed 1400 rows to /home/ridwan/projects/afrispeech-project/results/predictions/whisper_small_clinical_predictions.csv
  Flushed 1600 rows to /home/ridwan/projects/afrispeech-project/results/predictions/whisper_small_clinical_predictions.csv
  Flushed 1800 rows 

,audio_id,accent,reference,prediction,error
0,27a83595-3d3f-4a6b-b909-7b8f364d736b/7cb8dfb31...,setswana,Protection of the host immune mechanism mighti...,Protection of the host immune mechanism might...,
1,80581f4f-9e72-48fb-b4c0-22e6972dbff3/95ddccb7b...,setswana,Ask the patient about allergies totape and ski...,Ask the patients about allergies to tape and ...,
2,13f2d282-bda1-4bc3-83b8-7a0d06e5eda0/933137f33...,setswana,"The short, mucus-secreting cells between the c...",The short macro secreting cells between the c...,
3,99f6019b-4505-4c60-a462-79b0eb6e689a/2cb25e2dd...,setswana,Aspiration is terminated when aspirated materi...,6. Aspiration is terminated when aspirated ma...,
4,13600ba0-a4d5-46f7-9235-776cd60bfdee/1404f622c...,setswana,Identify the appropriate landmarks for the sit...,Identify the appropriate landmarks for the si...,
5,036df7a2-b9b1-47b0-8dc7-282ca51197e9/d169b1978...,setswana,Most ophthalmic drugs in general use are deliv...,Most ophthalmic drugs in general used are del...,
6,6ad08097-a797-4559-8bf3-b676af329695/2bd551f3f...,setswana,"However, cocaine, because of its abuse potenti...","However, koma, cocaine, koma, because of its ...",
7,f4cce4ae-63e5-4027-8521-f241310ed871/47a2370b2...,setswana,Check that packaged sterile drape is dry and u...,Check that package sterile drape is dry and o...,
8,5a46ea08-b98b-4f16-b1a2-aca74474e1a6/18f08d4af...,setswana,Even a small weight loss can significantly red...,Even a small weight loss can significantly re...,
9,21a67230-28d8-41e7-af2d-36533755c2cf/99cb4b64f...,setswana,Their onset of action is usually less than 20 ...,Their answer of action is usually less than 2...,


In [16]:
general_output_path = PREDICTIONS_DIR / "whisper_small_general_predictions.csv"

general_df = run_whisper_inference(
    dataset=general_ds,
    output_csv_path=general_output_path,
    model_name=MODEL_NAME,
    model_short_name=MODEL_SHORT_NAME,
    domain_name="general",
    max_samples=None,
    batch_size=32,
    flush_every=200
)

display(general_df[["audio_id", "accent", "reference", "prediction", "error"]].head(10))

Running inference on 2712 samples (batch_size=32)...


whisper_small | general:   0%|          | 0/2712 [00:00<?, ?it/s]

  Flushed 200 rows to /home/ridwan/projects/afrispeech-project/results/predictions/whisper_small_general_predictions.csv
  Flushed 400 rows to /home/ridwan/projects/afrispeech-project/results/predictions/whisper_small_general_predictions.csv
  Flushed 600 rows to /home/ridwan/projects/afrispeech-project/results/predictions/whisper_small_general_predictions.csv
  Flushed 800 rows to /home/ridwan/projects/afrispeech-project/results/predictions/whisper_small_general_predictions.csv
  Flushed 1000 rows to /home/ridwan/projects/afrispeech-project/results/predictions/whisper_small_general_predictions.csv
  Flushed 1200 rows to /home/ridwan/projects/afrispeech-project/results/predictions/whisper_small_general_predictions.csv
  Flushed 1400 rows to /home/ridwan/projects/afrispeech-project/results/predictions/whisper_small_general_predictions.csv
  Flushed 1600 rows to /home/ridwan/projects/afrispeech-project/results/predictions/whisper_small_general_predictions.csv
  Flushed 1800 rows to /home

,audio_id,accent,reference,prediction,error
0,27a83595-3d3f-4a6b-b909-7b8f364d736b/7cb8dfb31...,setswana,Protection of the host immune mechanism mighti...,Protection of the host immune mechanism might...,
1,80581f4f-9e72-48fb-b4c0-22e6972dbff3/95ddccb7b...,setswana,Ask the patient about allergies totape and ski...,Ask the patients about allergies to tape and ...,
2,13f2d282-bda1-4bc3-83b8-7a0d06e5eda0/933137f33...,setswana,"The short, mucus-secreting cells between the c...",The short macro secreting cells between the c...,
3,99f6019b-4505-4c60-a462-79b0eb6e689a/2cb25e2dd...,setswana,Aspiration is terminated when aspirated materi...,6. Aspiration is terminated when aspirated ma...,
4,13600ba0-a4d5-46f7-9235-776cd60bfdee/1404f622c...,setswana,Identify the appropriate landmarks for the sit...,Identify the appropriate landmarks for the si...,
5,036df7a2-b9b1-47b0-8dc7-282ca51197e9/d169b1978...,setswana,Most ophthalmic drugs in general use are deliv...,Most ophthalmic drugs in general used are del...,
6,6ad08097-a797-4559-8bf3-b676af329695/2bd551f3f...,setswana,"However, cocaine, because of its abuse potenti...","However, koma, cocaine, koma, because of its ...",
7,f4cce4ae-63e5-4027-8521-f241310ed871/47a2370b2...,setswana,Check that packaged sterile drape is dry and u...,Check that package sterile drape is dry and o...,
8,5a46ea08-b98b-4f16-b1a2-aca74474e1a6/18f08d4af...,setswana,Even a small weight loss can significantly red...,Even a small weight loss can significantly re...,
9,21a67230-28d8-41e7-af2d-36533755c2cf/99cb4b64f...,setswana,Their onset of action is usually less than 20 ...,Their answer of action is usually less than 2...,


In [17]:
# ── Cell 4: Model ─────────────────────────────────────────────────────────────
MODEL_NAME       = "openai/whisper-small.en"
MODEL_SHORT_NAME = "whisper_small_en"

device     = 0 if torch.cuda.is_available() else -1
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

asr_pipe = pipeline(
    task="automatic-speech-recognition",
    model=MODEL_NAME,
    torch_dtype=torch_dtype,
    device=device,
    chunk_length_s=30,
    stride_length_s=5,
)

print("Loaded:", MODEL_NAME)
print("Device:", "cuda" if device == 0 else "cpu")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Loaded: openai/whisper-small.en
Device: cuda


In [20]:
# ── Cell 5: Inference function ────────────────────────────────────────────────
def run_whisper_inference(
    dataset,
    output_csv_path,
    model_name,
    model_short_name,
    domain_name,
    max_samples=None,
    batch_size=32,
    flush_every=200
):
    output_csv_path = Path(output_csv_path)
    n_samples = len(dataset) if max_samples is None else min(max_samples, len(dataset))

    def audio_generator():
        for i in range(n_samples):
            sample = dataset[i]
            yield {
                "array": sample["audio"]["array"],
                "sampling_rate": sample["audio"]["sampling_rate"]
            }

    print("Collecting metadata...")
    meta_rows = []
    for i in range(n_samples):
        sample = dataset[i]
        meta_rows.append({
            "row_index":        i,
            "audio_id":         sample.get("audio_id", ""),
            "speaker_id":       sample.get("speaker_id", ""),
            "domain":           sample.get("domain", ""),
            "accent":           sample.get("accent", ""),
            "country":          sample.get("country", ""),
            "gender":           sample.get("gender", ""),
            "age_group":        sample.get("age_group", ""),
            "duration":         sample.get("duration", ""),
            "model_name":       model_name,
            "model_short_name": model_short_name,
            "reference":        sample.get("transcript", ""),
        })

    print(f"Running inference: {n_samples} samples | batch_size={batch_size}")
    start_time = time.time()

    predictions = []
    errors      = []

    results = asr_pipe(
        audio_generator(),
        batch_size=batch_size
    )

    for i, result in enumerate(tqdm(results, total=n_samples,
                                     desc=f"{model_short_name} | {domain_name}")):
        try:
            predictions.append(result["text"])
            errors.append("")
        except Exception as e:
            predictions.append("")
            errors.append(str(e))

        if (i + 1) % flush_every == 0:
            partial_df = pd.DataFrame(meta_rows[:i+1])
            partial_df["prediction"] = predictions
            partial_df["error"]      = errors
            partial_df.to_csv(output_csv_path, index=False)
            print(f"  Flushed {i+1} rows")

    df = pd.DataFrame(meta_rows)
    df["prediction"] = predictions
    df["error"]      = errors
    df.to_csv(output_csv_path, index=False)

    elapsed = round((time.time() - start_time) / 60, 2)
    print(f"\nDone. {len(df)} rows → {output_csv_path}")
    print(f"Elapsed: {elapsed} min | Errors: {df['error'].ne('').sum()}")

    return df

In [21]:
# ── Cell 6: Sanity check — 5 samples ─────────────────────────────────────────
test_df = run_whisper_inference(
    dataset=clinical_ds,
    output_csv_path=PREDICTIONS_DIR / "test_whisper_small_en_5samples.csv",
    model_name=MODEL_NAME,
    model_short_name=MODEL_SHORT_NAME,
    domain_name="clinical",
    max_samples=5,
    batch_size=4,
    flush_every=5
)

display(test_df[["audio_id", "accent", "reference", "prediction", "error"]])

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.


Running inference: 5 samples | batch_size=4


whisper_small_en | clinical:   0%|          | 0/5 [00:00<?, ?it/s]

  Flushed 5 rows

Done. 5 rows → /home/ridwan/projects/afrispeech-project/results/predictions/test_whisper_small_en_5samples.csv
Elapsed: 0.01 min | Errors: 0


,audio_id,accent,reference,prediction,error
0,27a83595-3d3f-4a6b-b909-7b8f364d736b/7cb8dfb31...,setswana,Protection of the host immune mechanism mighti...,Protection of the host immune mechanism might...,
1,80581f4f-9e72-48fb-b4c0-22e6972dbff3/95ddccb7b...,setswana,Ask the patient about allergies totape and ski...,Ask the patients about allergies to tape and ...,
2,13f2d282-bda1-4bc3-83b8-7a0d06e5eda0/933137f33...,setswana,"The short, mucus-secreting cells between the c...",The short marker secreting cells between the ...,
3,99f6019b-4505-4c60-a462-79b0eb6e689a/2cb25e2dd...,setswana,Aspiration is terminated when aspirated materi...,6. Aspiration is terminated when aspirated ma...,
4,13600ba0-a4d5-46f7-9235-776cd60bfdee/1404f622c...,setswana,Identify the appropriate landmarks for the sit...,Identify the appropriate landmarks for the si...,


In [22]:
# ── Cell 7: Full clinical run ─────────────────────────────────────────────────
clinical_en_df = run_whisper_inference(
    dataset=clinical_ds,
    output_csv_path=PREDICTIONS_DIR / "whisper_small_en_clinical_predictions.csv",
    model_name=MODEL_NAME,
    model_short_name=MODEL_SHORT_NAME,
    domain_name="clinical",
    max_samples=None,
    batch_size=32,
    flush_every=200
)

print("Clinical errors:", clinical_en_df["error"].ne("").sum())

Running inference: 3606 samples | batch_size=32


whisper_small_en | clinical:   0%|          | 0/3606 [00:00<?, ?it/s]

  Flushed 200 rows
  Flushed 400 rows
  Flushed 600 rows
  Flushed 800 rows
  Flushed 1000 rows
  Flushed 1200 rows
  Flushed 1400 rows
  Flushed 1600 rows
  Flushed 1800 rows
  Flushed 2000 rows
  Flushed 2200 rows
  Flushed 2400 rows
  Flushed 2600 rows
  Flushed 2800 rows
  Flushed 3000 rows
  Flushed 3200 rows
  Flushed 3400 rows
  Flushed 3600 rows

Done. 3606 rows → /home/ridwan/projects/afrispeech-project/results/predictions/whisper_small_en_clinical_predictions.csv
Elapsed: 10.92 min | Errors: 0
Clinical errors: 0


In [23]:
general_output_path = PREDICTIONS_DIR / "whisper_small_en_general_predictions.csv"

general_en_df = run_whisper_inference(
    dataset=general_ds,
    output_csv_path=general_output_path,
    model_name=MODEL_NAME,
    model_short_name=MODEL_SHORT_NAME,
    domain_name="general",
    max_samples=None,
    batch_size=32,
    flush_every=200
)

print("General errors:", general_en_df["error"].ne("").sum())
print("General samples:", len(general_en_df))

Running inference: 2712 samples | batch_size=32


whisper_small_en | general:   0%|          | 0/2712 [00:00<?, ?it/s]

  Flushed 200 rows
  Flushed 400 rows
  Flushed 600 rows
  Flushed 800 rows
  Flushed 1000 rows
  Flushed 1200 rows
  Flushed 1400 rows
  Flushed 1600 rows
  Flushed 1800 rows
  Flushed 2000 rows
  Flushed 2200 rows
  Flushed 2400 rows
  Flushed 2600 rows

Done. 2712 rows → /home/ridwan/projects/afrispeech-project/results/predictions/whisper_small_en_general_predictions.csv
Elapsed: 9.53 min | Errors: 0
General errors: 0
General samples: 2712
